# 12 — Chemistry Layer Ablation

From SUMMER_PLAN.md's Feature Ablation Study: systematically removes
individual chemistry-carrying inputs — residue-type embedding, and bond
edges (which also gates the bond-count node feature, so bond chemistry
can't leak back in through node embeddings on radial-only edges — see
`src/models/egnn.py` docstring) — and measures the impact on Pearson r and
RMSE. Identifies which chemistry inputs are load-bearing vs redundant.
Distinct from Phase 2 (notebook 08), which ablates *surface geometry*
(curvature/normal), not chemistry.

**Status: not yet run.** Depends on the staged ablation study (Phases 1-4,
notebooks 07, 08, 10, and 11) concluding first, so this runs against the
actual winning configuration rather than an arbitrary baseline.

## Ablation axes

Uses the `use_residue_embedding` / `use_bond_edges` / `use_radial_edges`
flags added to `DistanceESPN`/`AttentionESPN` (see `07_train.py --no-*`
flags). One-at-a-time removal from the full model, plus an all-off
stress-test config:

| Suffix | Residue embedding | Bond edges + bond count | Radial edges |
|--------|-------------------|--------------------------|---------------|
| `full` | on | on | on |
| `no_residue` | **off** | on | on |
| `no_bond` | on | **off** | on |
| `no_radial` | on | on | **off** |
| `chem_stripped` | **off** | **off** | **off** — only atom type + RBF distances remain |

## Prerequisites

- [ ] Staged ablation study (Phases 1-4) concluded, winning config known per architecture
- [ ] `sweeps/phase5_chemistry_ablation.yaml` written (5 configs x 2 models = 10 runs),
      locking in the winning agg/features/batching config from Phases 1-4
- [ ] Sweep run, `scripts/analyze_model.py --save-plots` run per checkpoint

## Decision

*To be filled in once this ablation completes.*

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path("../..").resolve()))


def show_png_grid(runs, filename, title, ncols=2):
    """Display saved PNG plots from each run's plot_dir in a grid."""
    available = [r for r in runs if (r["plot_dir"] / filename).exists()]
    if not available:
        print(f"No '{filename}' plots found. Run analyze_model.py --save-plots first.")
        return
    nrows = (len(available) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5 * nrows))
    axes = axes.flatten() if nrows * ncols > 1 else [axes]
    fig.suptitle(title, fontsize=14, fontweight="bold", y=1.01)
    for ax, run in zip(axes, available):
        ax.imshow(mpimg.imread(run["plot_dir"] / filename))
        ax.set_title(run["label"], fontsize=11)
        ax.axis("off")
    for ax in axes[len(available):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()


def plot_metric_bars(df, title_prefix, metrics):
    """1-row x 2-col grouped bar chart: one chart per model type."""
    if df.empty:
        print("No data to plot.")
        return
    model_types = ["Attention", "Distance"]
    fig, axes = plt.subplots(1, len(model_types), figsize=(13, 5), sharey=False)
    fig.suptitle(title_prefix, fontsize=13, fontweight="bold")
    n_metrics = len(metrics)
    total_width = 0.7
    bar_w = total_width / n_metrics
    for ax, model_type in zip(axes, model_types):
        sub = df[df["Model"] == model_type].copy()
        if sub.empty:
            ax.set_visible(False)
            continue
        x = range(len(sub))
        for i, (metric, color) in enumerate(metrics):
            if metric not in sub.columns or sub[metric].isna().all():
                continue
            offsets = [xi - total_width / 2 + bar_w * i + bar_w / 2 for xi in x]
            bars = ax.bar(offsets, sub[metric], width=bar_w, color=color, label=metric, zorder=3)
            ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=8, rotation=90)
        ax.set_title(model_type, fontsize=12)
        ax.set_xlabel("Chemistry config")
        ax.set_xticks(list(x))
        ax.set_xticklabels(sub["Config"].tolist(), fontsize=9, rotation=15, ha="right")
        ax.legend(fontsize=9)
        ax.grid(axis="y", alpha=0.3, zorder=0)
    plt.tight_layout()
    plt.show()

## 1. Configuration

In [ ]:
THESIS_ROOT = Path("/home/student/thesis")
CKPT_ROOT   = THESIS_ROOT / "checkpoints"
EVAL_ROOT   = THESIS_ROOT / "model_eval"

CONFIGS = ["full", "no_residue", "no_bond", "no_radial", "chem_stripped"]

RUNS = [
    dict(label=f"{model.capitalize()} — {cfg}", model_type=model, config=cfg,
         plot_dir=EVAL_ROOT/f"{model}_{cfg}", ckpt_dir=CKPT_ROOT/f"{model}_{cfg}")
    for model in ["attention", "distance"]
    for cfg in CONFIGS
]

print(f"{'Run':<28}  {'Plots':>6}  {'Metrics':>8}")
print("-" * 48)
for r in RUNS:
    has_plots   = r["plot_dir"].exists()
    has_metrics = (r["ckpt_dir"] / "metrics.csv").exists()
    print(f"{r['label']:<28}  {'yes' if has_plots else 'no':>6}  {'yes' if has_metrics else 'no':>8}")

## 2. Training Curves

In [ ]:
show_png_grid(RUNS, "training_curves.png", "Training Curves — Chemistry Ablation")

## 3. Error Distributions

In [ ]:
show_png_grid(RUNS, "error_distributions.png", "Error Distributions — Chemistry Ablation", ncols=1)

## 4. Validation Metrics Comparison (selection basis)

Same methodology as prior phases — best-val-loss epoch from `metrics.csv`.

In [ ]:
rows = []
for run in RUNS:
    csv_path = run["ckpt_dir"] / "metrics.csv"
    if not csv_path.exists():
        continue
    hist = pd.read_csv(csv_path)
    if hist.empty:
        continue
    best = hist.loc[hist["val_loss"].idxmin()]
    rows.append({
        "Run":           run["label"],
        "Model":         run["model_type"].capitalize(),
        "Config":        run["config"],
        "Pearson r":     best["val_pearson_r"],
        "RMSE":          best["val_rmse"],
        "Val loss":      best["val_loss"],
        "Train loss":    best["train_loss"],
        "Train/val gap": best["train_loss"] - best["val_loss"],
        "Best epoch":    int(best["epoch"]),
    })

val_df = pd.DataFrame(rows).sort_values(["Model", "Pearson r"], ascending=[True, False])
pd.set_option("display.float_format", "{:.4f}".format)
display(val_df)

In [ ]:
plot_metric_bars(val_df, "Validation metrics — Chemistry ablation (selection basis)",
                  metrics=[("Pearson r", "steelblue"), ("RMSE", "darkorange")])

## 5. Test Metrics (reference only — not used for selection)

In [ ]:
rows_test = []
for run in RUNS:
    metrics_path = run["ckpt_dir"] / "test_metrics.json"
    if not metrics_path.exists():
        continue
    with open(metrics_path) as f:
        data = json.load(f)
    g = data.get("global", {})
    rows_test.append({
        "Run":       run["label"],
        "Model":     run["model_type"].capitalize(),
        "Config":    run["config"],
        "Pearson r": g.get("pearson_r"),
        "RMSE":      g.get("rmse"),
        "MAE":       g.get("mae"),
        "N proteins": g.get("n_proteins"),
    })

test_df = pd.DataFrame(rows_test).sort_values(["Model", "Pearson r"], ascending=[True, False])
display(test_df)

In [ ]:
plot_metric_bars(test_df, "Test metrics — Chemistry ablation (reference only)",
                  metrics=[("Pearson r", "steelblue"), ("RMSE", "darkorange"), ("MAE", "seagreen")])

## 6. Error Distribution by ESP Value

Does stripping a chemistry input hurt uniformly, or specifically at extreme
ESP values where fine-grained charge context matters most? (SUMMER_PLAN.md
"Error Distribution by ESP Value".) Plots saved by
`scripts/analyze_model.py --error-by-esp` for each run.

Left panel: residual vs ground-truth ESP, coloured by net charge, with a
binned median |residual| trend line. Right panel: residual histograms split
by |ESP| tertile. `chem_stripped` vs `full` is the clearest comparison —
if the high-|ESP| tertile widens disproportionately once bond/residue
chemistry is removed, that's evidence the model leans on chemistry
specifically to resolve strongly-charged surface regions, not just as a
general accuracy boost.

In [ ]:
show_png_grid(RUNS, "error_by_esp.png", "Error Distribution by ESP Value — Chemistry Ablation", ncols=1)

## 7. Decision

*Fill in once this ablation completes.*

**Which chemistry inputs are load-bearing vs redundant?** — compare each
one-at-a-time removal against `full`. A large drop means that input is
load-bearing; little-to-no drop (or an improvement) means it's redundant
and a candidate for permanent removal to simplify the architecture.

**`chem_stripped` vs `full`** is the ceiling-to-floor spread for chemistry
inputs specifically — how much of the model's performance depends on
anything beyond atom type + geometry.